In [2]:
import duckdb

duckdb.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

In [3]:
duckdb.sql(f"""
        SELECT *
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        LIMIT 10
        """).show()

┌─────────┬─────────────────────┬───────────┬─────────┬─────────┬────────────────────┬─────────────┬───────────────┬────────────────┬─────────┬───────┬─────────┐
│   id    │        date         │ client_id │ card_id │ amount  │      use_chip      │ merchant_id │ merchant_city │ merchant_state │   zip   │  mcc  │ errors  │
│  int64  │      timestamp      │   int64   │  int64  │ varchar │      varchar       │    int64    │    varchar    │    varchar     │ double  │ int64 │ varchar │
├─────────┼─────────────────────┼───────────┼─────────┼─────────┼────────────────────┼─────────────┼───────────────┼────────────────┼─────────┼───────┼─────────┤
│ 7475327 │ 2010-01-01 00:01:00 │      1556 │    2972 │ $-77.00 │ Swipe Transaction  │       59935 │ Beulah        │ ND             │ 58523.0 │  5499 │ NULL    │
│ 7475328 │ 2010-01-01 00:02:00 │       561 │    4575 │ $14.57  │ Swipe Transaction  │       67570 │ Bettendorf    │ IA             │ 52722.0 │  5311 │ NULL    │
│ 7475329 │ 2010-01-01 00:02

### Contagem da Base 

vírgula dentro do valor quebra o parser CSV 

Original Line: 7500793,2010-01-07 12:07:00,1356,5480,$53.69,Swipe Transaction,83989,Colfax,WI,54730.0,5310,"Insufficient Balance,Technical Glitch"


In [ ]:

duckdb.sql(f"""
        SELECT COUNT(*) 
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        """).show()



duckdb.sql(f"""
        SELECT COUNT(*) 
        FROM read_csv_auto('s3://bronze/transactions_data.csv', ignore_errors=true)
        """).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     13305915 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     13304974 │
└──────────────┘



In [3]:

duckdb.sql(f"""
        DESC FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        """).show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ date           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ client_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ card_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ amount         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ use_chip       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ merchant_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ merchant_city  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ merchant_state │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ zip            │ DOUBLE      │ YES     │ NULL    

In [5]:
duckdb.sql(f"""
        SELECT DISTINCT errors
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        """).show()

duckdb.sql(f"""
        SELECT errors, COUNT(*)
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        GROUP BY errors
        ORDER BY 2 DESC 
        """).show()

┌───────────────────────┐
│        errors         │
│        varchar        │
├───────────────────────┤
│ Bad PIN               │
│ Bad Zipcode           │
│ NULL                  │
│ Bad Expiration        │
│ "Bad Zipcode          │
│ Bad CVV               │
│ "Bad Card Number      │
│ Technical Glitch      │
│ "Bad Expiration       │
│ Insufficient Balance  │
│ "Bad PIN              │
│ "Insufficient Balance │
│ Bad Card Number       │
│ "Bad CVV              │
└───────────────────────┘
         14 rows       

┌───────────────────────┬──────────────┐
│        errors         │ count_star() │
│        varchar        │    int64     │
├───────────────────────┼──────────────┤
│ NULL                  │     13094522 │
│ Insufficient Balance  │       130902 │
│ Bad PIN               │        32119 │
│ Technical Glitch      │        26271 │
│ Bad Card Number       │         7767 │
│ Bad Expiration        │         6161 │
│ Bad CVV               │         6106 │
│ Bad Zipcode           │     

In [8]:
duckdb.sql(f"""
        SELECT COUNT(DISTINCT merchant_city) Cidades , COUNT(merchant_city)
        FROM read_csv_auto('{base}/data/bronze/transactions_data.csv', ignore_errors=true)
        """).show()
# 12492 Cidades

┌─────────┬──────────────────────┐
│ Cidades │ count(merchant_city) │
│  int64  │        int64         │
├─────────┼──────────────────────┤
│   12492 │             13304974 │
└─────────┴──────────────────────┘



In [6]:
duckdb.sql(f"""
        SELECT COUNT(DISTINCT merchant_city) Cidade, COUNT(DISTINCT UPPER(TRIM(merchant_city))) Cidade
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        """).show()

duckdb.sql(f"""
        SELECT COUNT(DISTINCT merchant_state) Estado, COUNT(DISTINCT UPPER(TRIM(merchant_state))) Estado
        FROM read_csv_auto('s3://bronze/transactions_data.csv', strict_mode=false)
        """).show()

┌────────┬────────┐
│ Cidade │ Cidade │
│ int64  │ int64  │
├────────┼────────┤
│  12492 │  12492 │
└────────┴────────┘

┌────────┬────────┐
│ Estado │ Estado │
│ int64  │ int64  │
├────────┼────────┤
│    199 │    199 │
└────────┴────────┘



In [7]:
# Tentativa de entender o merchant_id 

duckdb.sql(f"""
        SELECT DISTINCT merchant_id , COUNT(merchant_city)
        FROM read_csv_auto('s3://bronze/transactions_data.csv', ignore_errors=true)
        GROUP BY merchant_id 
        HAVING COUNT(merchant_city) > 1
        ORDER BY 2 DESC
        LIMIT 110 
        """).show() 


┌─────────────┬──────────────────────┐
│ merchant_id │ count(merchant_city) │
│    int64    │        int64         │
├─────────────┼──────────────────────┤
│       59935 │               610023 │
│       27092 │               589039 │
│       61195 │               562375 │
│       39021 │               440206 │
│       43293 │               362822 │
│       22204 │               347489 │
│       14528 │               333502 │
│       60569 │               301633 │
│       50783 │               298219 │
│       75781 │               273346 │
│         ·   │                  ·   │
│         ·   │                  ·   │
│         ·   │                  ·   │
│       59474 │                15545 │
│        3558 │                14739 │
│       44795 │                14536 │
│       74934 │                14313 │
│       39991 │                14248 │
│       57133 │                14233 │
│       81759 │                14076 │
│       27310 │                14016 │
│       78644 │          

In [8]:
duckdb.sql(f"""
    SELECT
        merchant_id,
        COUNT(DISTINCT mcc) AS n_mcc,
        COUNT(DISTINCT merchant_city) AS n_cidades,
        COUNT(DISTINCT merchant_state) AS n_estados
    FROM read_csv_auto('s3://bronze/transactions_data.csv', ignore_errors=true)
    GROUP BY merchant_id 
    HAVING n_mcc > 1
    ORDER BY n_mcc DESC 
    """).show() 

┌─────────────┬───────┬───────────┬───────────┐
│ merchant_id │ n_mcc │ n_cidades │ n_estados │
│    int64    │ int64 │   int64   │   int64   │
├─────────────┼───────┼───────────┼───────────┤
│       24971 │     2 │         3 │         3 │
│       54499 │     2 │         3 │         3 │
│       48405 │     2 │         4 │         4 │
│       69462 │     2 │         4 │         4 │
│       33311 │     2 │         4 │         4 │
│       94555 │     2 │        16 │        14 │
│       14676 │     2 │         2 │         2 │
│       70281 │     2 │         3 │         3 │
│       71029 │     2 │         2 │         2 │
│       28978 │     2 │        13 │        11 │
│         ·   │     · │         · │         · │
│         ·   │     · │         · │         · │
│         ·   │     · │         · │         · │
│       31702 │     2 │         3 │         3 │
│       57869 │     2 │         2 │         2 │
│       90148 │     2 │         5 │         4 │
│       50506 │     2 │         2 │     